In [4]:
import copy

import kagglehub
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

path = kagglehub.competition_download("playground-series-s4e10")
print(path)

/Users/ignacio/.cache/kagglehub/competitions/playground-series-s4e10


In [5]:
# load the data
import pandas as pd

train_df = pd.read_csv(path + "/train.csv")

# print column names
print(train_df.columns)

Index(['id', 'person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'loan_status'],
      dtype='str')


In [6]:
train_df = train_df.drop(columns=["id"])
train_df = pd.get_dummies(train_df, columns=["person_home_ownership"], drop_first=True)
train_df = pd.get_dummies(train_df, columns=["loan_intent"], drop_first=True)
train_df = pd.get_dummies(train_df, columns=["loan_grade"], drop_first=True)
train_df["cb_person_default_on_file"] = train_df["cb_person_default_on_file"].map(
    {"Y": 1, "N": 0}
)
bool_cols = train_df.select_dtypes(include="bool").columns
train_df[bool_cols] = train_df[bool_cols].astype(int)

from sklearn.model_selection import train_test_split

X = train_df.drop(columns=["loan_status"])
y = train_df["loan_status"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Further split train into train/val for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

numeric_cols = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
]
scaler = StandardScaler()
X_tr[numeric_cols] = scaler.fit_transform(X_tr[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Device setup (MPS on Apple Silicon, else CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Class imbalance weight
pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f"Positive class weight: {pos_weight:.2f}")


class LoanDefaultNN(nn.Module):
    def __init__(self, input_size, hidden1_size):
        super(LoanDefaultNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1_size)
        self.bn1 = nn.BatchNorm1d(hidden1_size)
        self.fc2 = nn.Linear(hidden1_size, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.fc3 = nn.Linear(64, 32)
        self.bn3 = nn.BatchNorm1d(32)
        self.fc4 = nn.Linear(32, 16)
        self.bn4 = nn.BatchNorm1d(16)
        self.fc5 = nn.Linear(16, 8)
        self.bn5 = nn.BatchNorm1d(8)
        self.fc6 = nn.Linear(8, 4)
        self.bn6 = nn.BatchNorm1d(4)
        self.fc7 = nn.Linear(4, 2)
        self.bn7 = nn.BatchNorm1d(2)
        self.fc8 = nn.Linear(2, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # Dropout only on larger layers to avoid instability in small ones
        x = self.dropout(torch.relu(self.bn1(self.fc1(x))))
        x = self.dropout(torch.relu(self.bn2(self.fc2(x))))
        x = self.dropout(torch.relu(self.bn3(self.fc3(x))))
        x = torch.relu(self.bn4(self.fc4(x)))
        x = torch.relu(self.bn5(self.fc5(x)))
        x = torch.relu(self.bn6(self.fc6(x)))
        x = torch.relu(self.bn7(self.fc7(x)))
        x = self.fc8(x)  # No sigmoid — BCEWithLogitsLoss handles it
        return x


model = LoanDefaultNN(input_size=X_tr.shape[1], hidden1_size=128).to(device)
print(model)

# BCEWithLogitsLoss with class weight handles imbalance + numerically stable
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight], dtype=torch.float32).to(device)
)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    patience=5,
    factor=0.5,
)

X_tr_tensor = torch.tensor(X_tr.values, dtype=torch.float32)
y_tr_tensor = torch.tensor(y_tr.values, dtype=torch.float32).view(-1, 1)
X_val_tensor = torch.tensor(X_val.values, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1).to(device)

dataloader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(X_tr_tensor, y_tr_tensor),
    batch_size=1024,
    shuffle=True,
)

# Early stopping
best_val_loss = float("inf")
best_model_state = None
patience = 10
epochs_no_improve = 0
epochs = 200

for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        y_hat = model(X_batch)
        loss = criterion(y_hat, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_tensor)
        val_loss = criterion(val_logits, y_val_tensor).item()

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    print(
        f"Epoch [{epoch + 1}/{epochs}], Val Loss: {val_loss:.4f}, "
        f"Best: {best_val_loss:.4f}, No improve: {epochs_no_improve}",
        end="\r",
    )

    if epochs_no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch + 1}")
        break

# Load the best checkpoint
model.load_state_dict(best_model_state)
print(f"\nBest val loss: {best_val_loss:.4f}")

# Evaluate on hold-out test set
model.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32).to(device)
    y_test_tensor = (
        torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1).to(device)
    )
    logits = model(X_test_tensor)
    y_pred_proba = torch.sigmoid(logits).cpu().numpy().flatten()

y_test_np = y_test.values.astype(float)

# Find optimal threshold by maximizing F1 over a sweep
thresholds = np.arange(0.1, 0.9, 0.01)
f1s = [
    f1_score(y_test_np, (y_pred_proba >= t).astype(int), zero_division=0)
    for t in thresholds
]
optimal_threshold = thresholds[np.argmax(f1s)]
print(f"Optimal threshold: {optimal_threshold:.2f}")

y_pred_label = (y_pred_proba >= optimal_threshold).astype(float)
y_pred_tensor = torch.tensor(y_pred_label)
y_test_t = torch.tensor(y_test_np)

accuracy = (y_pred_tensor == y_test_t).float().mean()
precision = (y_pred_tensor * y_test_t).sum() / (y_pred_tensor.sum() + 1e-8)
recall = (y_pred_tensor * y_test_t).sum() / (y_test_t.sum() + 1e-8)
f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
auc = roc_auc_score(y_test_np, y_pred_proba)

print(f"Test Accuracy:  {accuracy.item():.4f}")
print(f"Test Precision: {precision.item():.4f}")
print(f"Test Recall:    {recall.item():.4f}")
print(f"Test F1 Score:  {f1.item():.4f}")
print(f"Test AUC-ROC:   {auc:.4f}")

Using device: mps
Positive class weight: 5.96
LoanDefaultNN(
  (fc1): Linear(in_features=22, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (bn3): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc4): Linear(in_features=32, out_features=16, bias=True)
  (bn4): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc5): Linear(in_features=16, out_features=8, bias=True)
  (bn5): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc6): Linear(in_features=8, out_features=4, bias=True)
  (bn6): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc7): Linear(in_features=4, out_features=2, 

In [9]:
test_df = pd.read_csv(path + "/test.csv")
test_df = pd.get_dummies(test_df, columns=["person_home_ownership"], drop_first=True)
test_df = pd.get_dummies(test_df, columns=["loan_intent"], drop_first=True)
test_df = pd.get_dummies(test_df, columns=["loan_grade"], drop_first=True)
test_df["cb_person_default_on_file"] = test_df["cb_person_default_on_file"].map(
    {"Y": 1, "N": 0}
)
bool_cols = test_df.select_dtypes(include="bool").columns
test_df[bool_cols] = test_df[bool_cols].astype(int)
test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

test_tensor = torch.tensor(test_df.drop(columns=["id"]).values, dtype=torch.float32).to(
    device
)

model.eval()
with torch.no_grad():
    test_logits = model(test_tensor)
    test_pred_proba = torch.sigmoid(test_logits).cpu().numpy().flatten()

test_pred_label = (test_pred_proba >= optimal_threshold).astype(float)
submission_df = pd.DataFrame({"id": test_df["id"], "loan_status": test_pred_label})
submission_df.to_csv("submission_loans.csv", index=False)
print(f"Saved {len(submission_df)} predictions (threshold={optimal_threshold:.2f})")

Saved 39098 predictions (threshold=0.80)


In [10]:
!kaggle competitions submit -c playground-series-s4e10 -f submission_loans.csv  -m "Message"

100%|█████████████████████████████████████████| 382k/382k [00:00<00:00, 551kB/s]
Successfully submitted to Loan Approval Prediction